## 02 약품 매칭 (복약)

OCR 또는 사용자 입력 약품명을 식약처 `drug_info`(43,276개 품목)와 매칭한다.

매칭 5단계 폴백 전략:

1. 정확 매칭 (`exact_match`) — 정규화 후 `_normalized` 1:1 lookup
2. Prefix 매칭 — 짧은 부분 입력 (예: '타이레놀') 본형 우선
3. 성분명 차단 — 단독 성분명 입력 시 임의 제품 매칭 차단
4. 모호 매칭 (`fuzzy_match`) — rapidfuzz WRatio + ATC tiebreak
5. 자모 분해 fuzzy — 한글 1~2자 오타 보정

신뢰도 분류:

| 점수 | 등급 | 처리 |
|---|---|---|
| 100 | exact | 자동 확정 |
| 90 이상 | high | 사용자 확인 후 확정 (1순위 자동 제안) |
| 70~90 | medium | 사용자 선택 (Top 3 후보 제시) |
| 70 미만 | low | 매칭 실패 → 직접 입력 |

정확 매칭 단일 결과(`exact_one`)일 때 자동 확정 경로로 분기하며, 동명이품 등 다중 매칭(`exact_multiple`)은 사용자 확인 분기를 거친다.

평가 결과 (30건 골든셋, v12): P@1 96.7%, P@5 100.0%, MRR 0.983

사용 데이터 (01_eda에서 캐시):

- `1_item_permit.pkl` (43,276행) — 약품 마스터
- `2_item_permit_detail.pkl` (43,276행) — ATC 코드 포함
- `5_easy_excel.pkl` (4,762행) — e약은요 (일반의약품)

## 환경 설정 및 라이브러리 임포트

In [ ]:
"""02_drug_matching - 약품명 매칭 (정확 + 모호 + 자모 분해).

OCR 또는 사용자 입력 약품명을 식약처 drug_info와 매칭하는 로직.
정확 매칭 1차, 실패 시 prefix 매칭(부분 입력), rapidfuzz WRatio 모호 매칭,
자모 분해 fuzzy(한국어 오타 보정) 순으로 폴백한다.
"""

# 표준 라이브러리
import os
import re
import subprocess
import sys

# 서드파티 라이브러리
import numpy as np
import pandas as pd

# rapidfuzz 설치 확인 (없으면 자동 설치)
try:
    import rapidfuzz
    print(f'rapidfuzz {rapidfuzz.__version__} 사용 가능')
except ImportError:
    print('rapidfuzz 설치 중...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rapidfuzz'])
    import rapidfuzz
    print(f'설치 완료: rapidfuzz {rapidfuzz.__version__}')

# jamo 설치 확인 (한국어 자모 분해 오타 보정용, v4 추가)
try:
    import jamo as _jamo_mod
    print('jamo 사용 가능')
except ImportError:
    print('jamo 설치 중...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'jamo'])
    import jamo as _jamo_mod
    print('설치 완료: jamo')

from rapidfuzz import fuzz, process
from jamo import h2j, j2hcj

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

## 환경 감지 및 경로 설정

01_eda와 동일한 환경 감지. 단 본 노트북은 pickle 캐시 디렉토리도 함께 참조한다.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/AH_03_06_medication_data'
    PROCESSED_DIR = '/content/drive/MyDrive/AH_03_06_medication_processed'
    ENV = 'Colab'
except ImportError:
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '../../data/raw/medication'))
    PROCESSED_DIR = os.path.abspath(os.path.join(os.getcwd(), '../../data/processed/medication'))
    ENV = 'Local'

print(f'환경: {ENV}')
print(f'원본 데이터: {BASE_DIR}')
print(f'캐시 데이터: {PROCESSED_DIR}')

assert os.path.isdir(PROCESSED_DIR), (
    f'pickle 캐시 디렉토리가 없습니다: {PROCESSED_DIR}\n'
    f'먼저 01_eda 노트북을 실행해 캐시를 생성하세요.'
)
print('캐시 디렉토리 확인됨')

## 데이터 로드 (pickle 캐시 우선)

02 작업에 필요한 데이터프레임 3개만 로드한다.

| 키 | 행수 | 용도 |
|---|---|---|
| 1_item_permit | 43,276 | 매칭 대상 약품 카탈로그 |
| 2_item_permit_detail | 43,276 | 후속 매칭 결과 보강용 (성분, 효능 등) |
| 5_easy_excel | 4,762 | 후속 일반의약품 친화 설명용 |

본 노트북에서는 1번만 본격 사용하고, 2·5번은 매칭 결과 enrichment 시 참조.

In [ ]:
# 02에 필요한 3개 데이터프레임만 pickle에서 빠르게 로드
NEEDED_KEYS = ['1_item_permit', '2_item_permit_detail', '5_easy_excel']

dfs = {}
for key in NEEDED_KEYS:
    pkl_path = os.path.join(PROCESSED_DIR, f'{key}.pkl')
    assert os.path.exists(pkl_path), f'pickle 파일 없음: {pkl_path}'

    df = pd.read_pickle(pkl_path)
    dfs[key] = df
    size_mb = os.path.getsize(pkl_path) / 1024**2
    print(f'  [{key}] {len(df):,}행 x {df.shape[1]}열 ({size_mb:.1f} MB)')

# 본 노트북의 메인 데이터프레임
df_drug = dfs['1_item_permit']
df_detail = dfs['2_item_permit_detail']
df_easy = dfs['5_easy_excel']

print(f'\n로드 완료. 총 {sum(len(df) for df in dfs.values()):,}행')

## 약품명 정규화 함수

OCR/사용자 입력의 표기 변동을 흡수하기 위한 정규화.

처리 항목:
- 괄호 안 내용 제거 (예: `타이레놀(아세트아미노펜)` -> `타이레놀`)
- 공백 제거
- 특수문자 제거 (한글·영문·숫자만 남김)
- 영문 소문자화

In [ ]:
def normalize_drug_name(name) -> str:
    """약품명에서 공백, 괄호 내용, 특수문자를 제거하고 소문자화한다.

    Args:
        name: 원본 약품명 (str 또는 NaN)

    Returns:
        정규화된 약품명. NaN/빈 문자열은 ''.
    """
    if pd.isna(name) or name is None:
        return ''
    name = str(name)

    # 1) 괄호 안 내용 제거: (수출명:...), (아세트아미노펜) 등
    name = re.sub(r'\([^)]*\)', '', name)
    # 2) 대괄호도 제거
    name = re.sub(r'\[[^\]]*\]', '', name)
    # 3) 단위 표기 통일 (밀리그람·그램 등 -> 영문 단위)
    for ko, en in [
        ('밀리그람', 'mg'), ('밀리그램', 'mg'), ('미리그람', 'mg'),
        ('마이크로그람', 'ug'), ('마이크로그램', 'ug'),
        ('밀리리터', 'ml'),
        ('그람', 'g'), ('그램', 'g'),
    ]:
        name = name.replace(ko, en)
    # 4) 공백 제거
    name = name.replace(' ', '').replace('\t', '')
    # 5) 한글·영문·숫자만 남기기
    name = re.sub(r'[^\w가-힣]', '', name)
    # 6) 영문 소문자화
    return name.lower()


# 정규화 테스트
test_normalize = [
    '타이레놀정 500mg',
    '게보린정 (수출명:돌로린정)',
    '아모잘탄정 5/50mg',
    '중외5%포도당생리식염액(수출명:5%DextroseinnormalsalineInj.)',
    '가스디알정50밀리그램(디메크로틴산마그네슘)',
    None,
    '',
]
print('정규화 테스트:')
for t in test_normalize:
    result = normalize_drug_name(t)
    print(f"  {repr(t):60s} -> {repr(result)}")

## 정확 매칭 함수

정규화된 입력과 정규화된 품목명을 비교해 일치하는 약을 찾는다.

**반환 형식**:
- 단일 매칭: `{'status': 'exact_one', 'confidence': 100, 'drug_info': {...}}`
- 동명이품(여러 매칭): `{'status': 'exact_multiple', 'confidence': 100, 'candidates': [...]}`
- 매칭 실패: `None`

미리 `_normalized` 컬럼을 생성해두면 매번 재계산 비용 절감.

In [ ]:
# 성능을 위해 _normalized 컬럼 미리 생성 (43,276행 1회만)
print('품목명 정규화 컬럼 생성 중 ...')
df_drug['_normalized'] = df_drug['품목명'].apply(normalize_drug_name)
print(f'완료. _normalized unique 개수: {df_drug["_normalized"].nunique():,}')


def jamo_decompose(text: str) -> str:
    """한글 텍스트를 호환 자모로 분해한다 (v4 추가).

    한국어 1~2글자 오타 보정에 효과적. character level fuzzy로는
    완전히 다른 글자로 처리되는 '부 vs 보' 같은 케이스를 자모 1자 차이로 잡아낸다.

    예: '게보린' -> 'ㄱㅔㅂㅗㄹㅣㄴ' (7자모)
        '게부린' -> 'ㄱㅔㅂㅜㄹㅣㄴ' (7자모, 모음 ㅗ -> ㅜ 1자 차이)

    한글이 아닌 문자(영문, 숫자, 특수문자)는 그대로 유지한다.

    Args:
        text: 정규화된 약품명 (한글 + 영문 + 숫자)

    Returns:
        호환 자모로 분해된 문자열. 변환 실패 시 원본 반환.
    """
    if not text:
        return ''
    try:
        return j2hcj(h2j(text))
    except Exception:
        # 자모 분해 불가능한 문자(특수문자 등) 포함 시 원본 반환
        return text


# 성능을 위해 _jamo 컬럼도 미리 생성 (43,276행 1회만, v4 추가)
print('품목명 자모 분해 컬럼 생성 중 ...')
df_drug['_jamo'] = df_drug['_normalized'].apply(jamo_decompose)
print(f'완료. _jamo unique 개수: {df_drug["_jamo"].nunique():,}')


def exact_match(query: str, df: pd.DataFrame) -> dict | None:
    """정규화된 query를 _normalized 컬럼과 정확 매칭한다.

    Args:
        query: 사용자 입력 약품명
        df: drug_info 데이터프레임 (_normalized 컬럼 보유)

    Returns:
        단일 매칭: {'status': 'exact_one', 'confidence': 100, 'drug_info': dict}
        여러 매칭: {'status': 'exact_multiple', 'confidence': 100, 'candidates': list}
        매칭 실패: None
    """
    normalized_query = normalize_drug_name(query)
    if not normalized_query:
        return None

    matches = df[df['_normalized'] == normalized_query]

    if len(matches) == 0:
        return None

    # v4: 내부용 컬럼(_normalized, _jamo)을 결과 dict에서 제거
    if len(matches) == 1:
        return {
            'status': 'exact_one',
            'confidence': 100,
            'drug_info': matches.iloc[0].drop(['_normalized', '_jamo'], errors='ignore').to_dict(),
        }

    # 동명이품: 같은 이름 다른 제조사 등
    return {
        'status': 'exact_multiple',
        'confidence': 100,
        'candidates': matches.drop(columns=['_normalized', '_jamo'], errors='ignore').head(5).to_dict('records'),
    }


# 정확 매칭 간단 테스트
print('\n정확 매칭 테스트:')
for q in ['타이레놀정500mg', '존재하지않는약XYZ']:
    r = exact_match(q, df_drug)
    if r is None:
        print(f"  '{q}' -> 매칭 실패")
    else:
        print(f"  '{q}' -> {r['status']}, confidence={r['confidence']}")


# ============================================================
# v8 (PR11): 성분명 입력 감지 및 차단
# - v7은 영문 '주성분' 컬럼 기반이라 한글 query lookup 불가 (실패).
# - v8: 품목명 괄호 안 한글 성분명 추출로 변경 (probe5 검증 완료).
#   핵심 성분 6/6 잡힘, brand 11/11 false positive 없음.
# ============================================================

import re as _re_ing

# 품목명 괄호 안 추출 (중첩 없는 단순 케이스)
_PAREN_PATTERN = _re_ing.compile(r'\(([^()]+)\)')

# 한글 + 일부 구분자만 허용 (영문/숫자/% 포함 시 제외)
# 허용: 한글, 공백, |, /, ·, -
_HANGUL_INGREDIENT = _re_ing.compile(r'^[가-힣][가-힣\s|/·\-]*$')

# 염/산/수화물 suffix
_SALT_SUFFIX_PATTERN = _re_ing.compile(
    r'(염산염|베실산염|말레산염|메탄설폰산염|시트르산염|구연산염|'
    r'인산염|황산염|타르타르산염|푸마르산염|글루콘산염|아세트산염|'
    r'칼륨|나트륨|칼슘|마그네슘|수화물|일수화물|이수화물|삼수화물)+$'
)


def _extract_ingredient_base(s: str) -> str:
    """염/산/수화물 suffix 반복 제거해 INN base 추출.

    예: '암로디핀베실산염' -> '암로디핀'
        '시타글립틴인산염수화물' -> '시타글립틴'
    """
    if not isinstance(s, str):
        return ''
    return _SALT_SUFFIX_PATTERN.sub('', s).strip()


# 성분명 set 구축 (품목명 괄호 안 한글, 43,276행 1회만)
print('성분명 set 구축 중 (품목명 괄호 안 한글) ...')
INGREDIENT_SET = set()
for name in df_drug['품목명'].dropna():
    name = str(name)
    for paren in _PAREN_PATTERN.findall(name):
        # 수출명 표기 제외 (예: '게보린정(수출명:돌로린정)')
        if paren.startswith('수출'):
            continue
        # 복합제 | 로 split
        for part in paren.split('|'):
            part = part.strip()
            if not part:
                continue
            # 한글 only (영문 brand 이름 등 제외)
            if not _HANGUL_INGREDIENT.match(part):
                continue
            INGREDIENT_SET.add(part)
            base = _extract_ingredient_base(part)
            if base and base != part:
                INGREDIENT_SET.add(base)
print(f'완료. INGREDIENT_SET: {len(INGREDIENT_SET):,}개')


# 제형 suffix (성분명+제형 패턴 인식용)
_DOSAGE_FORM_SUFFIXES = ('정', '캡슐', '시럽', '주사', '주', '액', '연질캡슐', '경질캡슐', '서방정')

# 용량 패턴 (NNmg, NN.NNmg, NN.NNg, NN%)
_DOSE_PATTERN = _re_ing.compile(r'\d+(\.\d+)?(mg|g|ml|%|밀리그램|밀리그람)?$')


def is_likely_ingredient_query(query: str) -> bool:
    """query가 성분명/성분명+제형/성분명+용량 입력인지 판정 (v8).

    True면 match_drug에서 best_match=None 반환 (PR11 정책).

    인식 패턴:
        1. 정규화 query 자체가 성분명 (예: '아세트아미노펜', '암로디핀')
        2. 정규화 query에서 trailing 용량 제거 후 성분명
        3. 정규화 query에서 trailing 제형(+용량) 제거 후 성분명
           (예: '메트포르민정500mg' -> '메트포르민')

    Args:
        query: 사용자 입력 원본 (정규화 전).

    Returns:
        True면 성분명 입력으로 판정 (best_match=None 권장).
    """
    if not query:
        return False
    nq = normalize_drug_name(query)
    if not nq:
        return False

    # 패턴 1: 정규화 query 자체
    if nq in INGREDIENT_SET:
        return True

    # 패턴 2: trailing 용량 제거 후
    nq_no_dose = _DOSE_PATTERN.sub('', nq).strip()
    if nq_no_dose and nq_no_dose != nq and nq_no_dose in INGREDIENT_SET:
        return True

    # 패턴 3: trailing 제형(+용량) 제거 후
    for candidate in (nq, nq_no_dose):
        if not candidate:
            continue
        for suf in _DOSAGE_FORM_SUFFIXES:
            if candidate.endswith(suf):
                base = candidate[:-len(suf)].strip()
                if base and base in INGREDIENT_SET:
                    return True

    return False


# 성분명 차단 간단 테스트
print('\n성분명 입력 감지 테스트:')
for q in ['아세트아미노펜', '암로디핀', '메트포르민정 500mg',
          '타이레놀정500mg', '게보린', '닥터베아제정']:
    flag = is_likely_ingredient_query(q)
    print(f"  '{q}' -> {'[성분명]' if flag else '[brand]'}")


# ============================================================
# v9 추가: ATC_MAP 빌드 (PR12 ATC tiebreak 용)
# ============================================================
# df_drug ↔ df_detail 조인 키: 품목일련번호 (int64)
# 같은 ATC를 공유하는 후보들 중 brand 일치도가 높은 것을 우선시 (변종 vs 본형 구분)
print('ATC_MAP 구축 중 (df_detail.ATC코드 기반) ...')

ATC_MAP = {}
for _item_no, _atc in zip(df_detail['품목일련번호'], df_detail['ATC코드']):
    if pd.notna(_atc) and str(_atc).strip():
        try:
            ATC_MAP[int(_item_no)] = str(_atc).strip()
        except (ValueError, TypeError):
            continue

print(f'완료. ATC_MAP 크기: {len(ATC_MAP):,}개')
print(f'  ATC 보유 비율: {len(ATC_MAP) / len(df_detail):.1%}')

# 검증: 4건 실패 케이스 ATC 확인
print('\n[v9 검증] 4건 실패 케이스 ATC:')
_CASES = [
    (202106092, '타이레놀정500밀리그람'),
    (202200658, '우먼스타이레놀정'),
    (200300406, '닥터베아제정'),
    (198700405, '베아제정'),
    (202200407, '타이레놀이알서방정'),
    (200108763, '디오반필름코팅정80mg'),
    (200809324, '디오르반정80mg'),
]
for _item_no, _name in _CASES:
    _atc = ATC_MAP.get(_item_no, '(missing)')
    print(f'  {_item_no} ({_name}): ATC={_atc!r}')


## 모호 매칭 함수 (rapidfuzz WRatio)

정확 매칭이 실패했을 때 사용. 정규화된 입력과 정규화된 모든 품목명의 유사도를 계산.

**WRatio**: rapidfuzz의 가중 비율 알고리즘. 부분 일치, 토큰 일치, 정렬 일치를 종합.

**임계값 정책**:
- 90 이상: high (자동 1순위 제안)
- 70~90: medium (Top 3 후보 제시)
- 70 미만: low (매칭 실패로 간주)

**성능**: 43K개 품목 x WRatio 비교는 약 1~2초.

In [ ]:
def _classify_confidence(score: float) -> str:
    """rapidfuzz 점수를 신뢰도 등급으로 분류한다."""
    if score >= 90:
        return 'high'
    if score >= 70:
        return 'medium'
    return 'low'


# 동점 처리에 사용되는 한정어 prefix (사용자가 일반 제품을 의도할 가능성 높을 때 후순위로)
QUALIFIER_PREFIXES = ('어린이', '소아용', '소아', '노인용')

# 변종 식별 키워드 - 약명 어디에든 들어가면 본형보다 후순위
# (콜드/쿨다운/릴랙스 등 동일 브랜드의 변종 제품. 사용자가 단순 입력 시 본형 우선)
VARIANT_KEYWORDS = (
    '콜드', '쿨다운', '릴랙스', '에스', '이알', '서방',
    '츄어블', '구강붕해', '액', '시럽', '연질', '플러스',
    '듀오', '포르테', '맥스', '나이트', '데이', '키즈',
    # v3 추가: 한국 약품 시장 흔한 변종 suffix
    '브이', '알파', '골드', '슈퍼', '액티브', '울트라', '프리미엄',
)


def fuzzy_match(query: str, df: pd.DataFrame, top_n: int = 5) -> list[dict]:
    """정규화된 query와 _normalized 컬럼의 유사도를 계산해 상위 N개 반환한다.

    동점 점수일 때 다음 우선순위로 1순위를 결정한다:
        1. 정규화 매칭 텍스트가 query로 시작하는 후보
        2. 한정어 prefix(어린이/소아용 등) 없는 후보
        3. 수출용 전용(`(수출용)`) 아닌 후보
           ※ `(수출명:...)` 표기는 한국 본형의 별칭이므로 후순위 X
        4. 변종 키워드(콜드/쿨다운/브이 등) 수가 적은 후보 (본형 우선)
        5. 짧은 품목명

    Args:
        query: 사용자 입력 약품명
        df: drug_info 데이터프레임 (_normalized 컬럼 보유)
        top_n: 반환할 후보 개수 (기본 5)

    Returns:
        list of dict: [{'score': float, 'confidence_level': str, 'drug_info': dict}, ...]
        점수 내림차순 + tiebreak 적용. 빈 리스트면 입력이 비어있음.
    """
    normalized_query = normalize_drug_name(query)
    if not normalized_query:
        return []

    # 동점 후보가 top_n 밖에 있을 수 있어 더 넉넉히 받아 tiebreak 후 자름
    fetch_n = max(top_n * 3, 15)
    normalized_choices = df['_normalized'].tolist()
    results = process.extract(
        normalized_query,
        normalized_choices,
        scorer=fuzz.WRatio,
        limit=fetch_n,
    )

    candidates = []
    for matched_text, score, idx in results:
        # v4: 내부용 컬럼(_normalized, _jamo)을 결과 dict에서 제거
        drug_row = df.iloc[idx].drop(['_normalized', '_jamo'], errors='ignore').to_dict()
        name = drug_row['품목명']
        # 동점 처리용 보조 키 (sort ascending에서 작을수록 우선)
        not_starts_with_query = not matched_text.startswith(normalized_query)
        has_qualifier = any(name.startswith(q) for q in QUALIFIER_PREFIXES)
        # (수출용)만 후순위. (수출명:...)은 본형의 별칭이므로 후순위 X
        is_export_only = '(수출용)' in name
        variant_count = sum(1 for kw in VARIANT_KEYWORDS if kw in name)
        candidates.append({
            'score': round(score, 1),
            'confidence_level': _classify_confidence(score),
            'drug_info': drug_row,
            '_tiebreak': (
                not_starts_with_query,
                has_qualifier,
                is_export_only,
                variant_count,
                len(name),
            ),
        })

    # 동점 시: query prefix 매칭 -> 한정어 없음 -> 비수출전용 -> 변종 적음 -> 짧은 이름 순
    candidates.sort(key=lambda c: (-c['score'], c['_tiebreak']))

    # 내부용 키는 결과에서 제거
    for c in candidates:
        del c['_tiebreak']

    return candidates[:top_n]


def fuzzy_match_jamo(query: str, df: pd.DataFrame, top_n: int = 5) -> list[dict]:
    """자모 분해 후 fuzzy 매칭한다 (v4 추가, 한국어 오타 보정용).

    fuzzy_match가 character level에서 잡지 못하는 한국어 1~2글자 오타를
    잡아내기 위한 보조 매칭. 자모 분해 후 rapidfuzz WRatio로 비교.

    예: '게부린' (사용자 오타) -> 'ㄱㅔㅂㅜㄹㅣㄴ'
        '게보린' (DB 정답)    -> 'ㄱㅔㅂㅗㄹㅣㄴ'
        자모 1자 차이 -> WRatio 점수 매우 높게 나옴

    Args:
        query: 사용자 입력 약품명
        df: drug_info 데이터프레임 (_normalized, _jamo 컬럼 보유)
        top_n: 반환할 후보 개수

    Returns:
        list of dict: 동일 형식 (score, confidence_level, drug_info)
        점수 내림차순 + tiebreak 적용. 빈 리스트면 매칭 없음.
    """
    normalized_query = normalize_drug_name(query)
    if not normalized_query:
        return []

    jamo_query = jamo_decompose(normalized_query)
    if not jamo_query:
        return []

    if '_jamo' not in df.columns:
        # _jamo 컬럼이 없으면 매칭 불가 (셀 10에서 미리 생성되어 있어야 함)
        return []

    fetch_n = max(top_n * 3, 15)
    jamo_choices = df['_jamo'].tolist()
    results = process.extract(
        jamo_query,
        jamo_choices,
        scorer=fuzz.WRatio,
        limit=fetch_n,
    )

    candidates = []
    for matched_text, score, idx in results:
        drug_row = df.iloc[idx].drop(['_normalized', '_jamo'], errors='ignore').to_dict()
        name = drug_row['품목명']
        not_starts_with_query = not matched_text.startswith(jamo_query)
        has_qualifier = any(name.startswith(q) for q in QUALIFIER_PREFIXES)
        is_export_only = '(수출용)' in name
        variant_count = sum(1 for kw in VARIANT_KEYWORDS if kw in name)
        candidates.append({
            'score': round(score, 1),
            'confidence_level': _classify_confidence(score),
            'drug_info': drug_row,
            '_tiebreak': (
                not_starts_with_query,
                has_qualifier,
                is_export_only,
                variant_count,
                len(name),
            ),
        })

    candidates.sort(key=lambda c: (-c['score'], c['_tiebreak']))
    for c in candidates:
        del c['_tiebreak']

    return candidates[:top_n]


# 모호 매칭 간단 테스트
print('모호 매칭 테스트:')
for q in ['타이레놀', '게보린']:
    cands = fuzzy_match(q, df_drug, top_n=3)
    print(f"\n  '{q}' Top 3 (character level):")
    for c in cands:
        print(f"    {c['score']:5.1f}/100 [{c['confidence_level']:6s}] {c['drug_info']['품목명']}")

print('\n자모 분해 fuzzy 테스트 (오타 케이스):')
for q in ['게부린정']:
    cands = fuzzy_match_jamo(q, df_drug, top_n=3)
    print(f"\n  '{q}' Top 3 (jamo level):")
    for c in cands:
        print(f"    {c['score']:5.1f}/100 [{c['confidence_level']:6s}] {c['drug_info']['품목명']}")

## 통합 매칭 함수 (정확 -> 모호 폴백)

실제 사용 진입점. 정확 매칭 시도 후 실패 시 모호 매칭으로 폴백.

**반환 구조**:
```
{
    'query': '...',
    'match_type': 'exact_one' | 'exact_multiple' | 'fuzzy',
    'confidence': int,
    'best_match': dict | None,
    'all_candidates': list[dict],
}
```

In [ ]:
# 짧은 query를 부분 입력으로 판정하는 길이 임계값 (정규화 후 글자 수)
SHORT_QUERY_THRESHOLD = 5

# v4: 자모 fuzzy 트리거 조건
# fuzzy 점수가 이 미만이면 자모 분해 fuzzy 추가 시도 (오타 의심)
JAMO_TRIGGER_THRESHOLD = 85
# 자모 fuzzy 결과는 최소 이 점수 이상이어야 채택 (낮은 점수 채택 시 거짓 매칭 위험)
JAMO_MIN_ACCEPT_SCORE = 70

# v12 (PR13): fuzzy_match가 fetch하는 후보 풀 크기.
# match_drug 외부 시그니처(fuzzy_top_n=5)는 반환 후보 수이며,
# 내부적으로 더 큰 풀에서 ATC tiebreak로 재정렬한 뒤 잘라낸다.
# 5에서 15로 늘려 본형이 변종 뒤에 묻혀 누락되는 케이스(예: 디오반정 80mg) 회수.
FETCH_TOP_N = 15


def _prefix_match(normalized_query: str, df: pd.DataFrame, top_n: int = 5) -> list[dict]:
    """정규화된 query로 시작하는 후보를 추출하고 본형 우선 정렬한다.

    부분 입력(짧은 query) 케이스 전용. WRatio가 자르는 후보 풀에서 누락되는
    문제를 피하고, 변종(콜드/쿨다운 등)보다 본형(브랜드명+제형)을 1순위로 끌어올린다.

    예: '타이레놀' -> '타이레놀'로 시작하는 모든 약 중
        변종 키워드 적은 + 짧은 이름 우선.

    Args:
        normalized_query: 정규화된 사용자 입력
        df: drug_info 데이터프레임 (_normalized 컬럼 보유)
        top_n: 반환할 후보 개수

    Returns:
        list of dict: prefix 매칭 + tiebreak 적용된 후보. 빈 리스트면 매칭 없음.
    """
    starts_with = df[df['_normalized'].str.startswith(normalized_query)]
    if len(starts_with) == 0:
        return []

    candidates = []
    for _, row in starts_with.iterrows():
        drug_row = row.drop(['_normalized', '_jamo'], errors='ignore').to_dict()
        name = drug_row['품목명']
        has_qualifier = any(name.startswith(q) for q in QUALIFIER_PREFIXES)
        is_export_only = '(수출용)' in name
        variant_count = sum(1 for kw in VARIANT_KEYWORDS if kw in name)
        candidates.append({
            'score': 95.0,
            'confidence_level': 'high',
            'drug_info': drug_row,
            '_tiebreak': (has_qualifier, is_export_only, variant_count, len(name)),
        })

    candidates.sort(key=lambda c: c['_tiebreak'])
    for c in candidates:
        del c['_tiebreak']

    return candidates[:top_n]


def _common_chunk_score(query_normalized: str, candidate_normalized: str) -> float:
    """쿼리 음절이 candidate에 등장하는 비율을 0~10점으로 환산.

    v9 (PR12): ATC tiebreak에서 brand 일치도 측정용.
    한국 약품명은 brand prefix가 보존되는 특성을 활용.
    음절 set 기반이라 순서 무관 — 변형(접두/접미 추가)에 강함.

    예시:
        '닥타베아제정' (6자) vs '닥터베아제정' (6자) -> 5/6 = 0.83 -> 8.3
        '닥타베아제정' (6자) vs '베아제정' (4자)   -> 4/6 = 0.67 -> 6.7
        '타이레놀이알' (6자) vs '타이레놀정500밀리그람(아세트아미노펜)' -> 4/6 = 0.67 -> 6.7
        '타이레놀이알' (6자) vs '타이레놀8시간이알서방정(아세트아미노펜)' -> 6/6 = 1.00 -> 10.0

    Args:
        query_normalized: 정규화된 쿼리 (한글 음절 단위).
        candidate_normalized: 정규화된 candidate 품목명.

    Returns:
        0.0 ~ 10.0 점수.
    """
    if not query_normalized:
        return 0.0
    matches = sum(1 for ch in query_normalized if ch in candidate_normalized)
    return (matches / len(query_normalized)) * 10.0


def _common_prefix_len(s1: str, s2: str) -> int:
    """두 정규화 문자열의 공통 prefix 음절 길이.

    v11 (PR12 강화): ATC tiebreak에서 brand 식별력 추가 (chunk score 보완).
    예: '닥타베아제정' vs '닥터베아제정' -> 1 ('닥' 일치, '타'≠'터')
        '타이레놀이알' vs '타이레놀정500밀리그람' -> 4 ('타이레놀' 일치)
        '닥타베아제정' vs '베아제정' -> 0
    """
    n = 0
    for a, b in zip(s1, s2):
        if a == b:
            n += 1
        else:
            break
    return n



def _atc_tiebreak(results: list[dict], atc_map: dict, query_normalized: str,
                  top_n: int = 5) -> list[dict]:
    """동일 ATC 후보군에 brand chunk bonus를 더해 재정렬한다.

    v9 (PR12): fuzzy/fuzzy_jamo 결과에 후처리 적용.
    1순위 후보의 ATC를 anchor로 삼아, 동일 ATC 후보들의 chunk 일치도 보너스만큼
    가산 후 재정렬. 원본 score는 보존(`_rerank_score` 키로 분리).

    Args:
        results: fuzzy_match/fuzzy_match_jamo 결과 (score, drug_info 포함).
        atc_map: dict[품목일련번호(int) -> ATC코드(str)].
        query_normalized: 정규화된 쿼리.
        top_n: 반환 후보 개수.

    Returns:
        재정렬된 결과 리스트. anchor ATC 없으면 원본 그대로 반환.
        각 항목에 `_atc_bonus` 키 추가 (디버깅·검증용).
    """
    if len(results) <= 1 or not query_normalized:
        return results

    # 1순위 후보의 ATC를 anchor로 선정
    top_item_no = results[0]['drug_info'].get('품목일련번호')
    try:
        anchor_atc = atc_map.get(int(top_item_no))
    except (ValueError, TypeError):
        return results
    if not anchor_atc:  # NaN/None/빈 문자열이면 tiebreak 적용 안 함
        return results

    # 동일 ATC 후보에만 chunk bonus 적용
    rescored = []
    for r in results:
        item_no = r['drug_info'].get('품목일련번호')
        try:
            c_atc = atc_map.get(int(item_no))
        except (ValueError, TypeError):
            c_atc = None

        bonus = 0.0
        if c_atc == anchor_atc:
            # v10 fix: fuzzy_match가 _normalized를 drop하므로 품목명을 재정규화
            cand_name = r['drug_info'].get('품목명', '')
            cand_norm = normalize_drug_name(cand_name) if cand_name else ''
            # v11→v12: chunk + prefix×20 (디오반 케이스의 fuzzy 격차 14점 극복)
            _chunk = _common_chunk_score(query_normalized, cand_norm)
            _prefix = _common_prefix_len(query_normalized, cand_norm)
            bonus = _chunk + _prefix * 20

        new_r = dict(r)
        new_r['_atc_bonus'] = bonus
        new_r['_rerank_score'] = r['score'] + bonus  # 원본 score 보존
        rescored.append(new_r)

    rescored.sort(key=lambda x: -x['_rerank_score'])
    return rescored[:top_n]


def match_drug(query: str, df: pd.DataFrame, fuzzy_top_n: int = 5) -> dict:
    """약품명 매칭의 통합 인터페이스.

    1단계: 정확 매칭 시도
    1.5단계: 짧은 query(부분 입력) -> prefix 정확 매칭 우선
    1.7단계 [v7]: 성분명/성분명+제형 입력 -> best_match=None 강제 (PR11)
    2단계: 모호 매칭 폴백 (rapidfuzz WRatio, character level)
    2.2단계 [v9]: ATC 기반 tiebreak (PR12) — fuzzy 결과 재정렬
    2.5단계 [v4]: fuzzy 점수 낮으면 자모 분해 fuzzy 추가 시도
    2.7단계 [v9]: jamo 결과에도 ATC tiebreak 적용

    Args:
        query: 사용자 입력 약품명
        df: drug_info 데이터프레임
        fuzzy_top_n: 모호 매칭 시 후보 개수

    Returns:
        dict with keys:
            - query: 원본 입력
            - match_type: 'exact_one'|'exact_multiple'|'prefix'|
                          'ingredient_blocked'|'fuzzy'|'fuzzy_jamo'
            - confidence: 100|95|0|fuzzy 점수|자모 fuzzy 점수
            - best_match: 1순위 후보 (또는 None)
            - all_candidates: 전체 후보 리스트
    """
    # 1단계: 정확 매칭
    exact = exact_match(query, df)

    if exact is not None and exact['status'] == 'exact_one':
        return {
            'query': query,
            'match_type': 'exact_one',
            'confidence': 100,
            'best_match': exact['drug_info'],
            'all_candidates': [exact['drug_info']],
        }

    if exact is not None and exact['status'] == 'exact_multiple':
        return {
            'query': query,
            'match_type': 'exact_multiple',
            'confidence': 100,
            'best_match': exact['candidates'][0],
            'all_candidates': exact['candidates'],
        }

    # 1.5단계: 짧은 query는 prefix 정확 매칭을 fuzzy보다 먼저 시도
    normalized_query = normalize_drug_name(query)
    if 0 < len(normalized_query) <= SHORT_QUERY_THRESHOLD:
        prefix_results = _prefix_match(normalized_query, df, top_n=fuzzy_top_n)
        if prefix_results:
            return {
                'query': query,
                'match_type': 'prefix',
                'confidence': prefix_results[0]['score'],
                'best_match': prefix_results[0]['drug_info'],
                'all_candidates': [c['drug_info'] for c in prefix_results],
            }

    # 1.7단계 (v7): 성분명/성분명+제형 입력은 fuzzy 단계 전에 차단
    # 임의 회사 제품을 1순위로 주는 의학적 위험 방지 (PR11)
    if is_likely_ingredient_query(query):
        return {
            'query': query,
            'match_type': 'ingredient_blocked',
            'confidence': 0,
            'best_match': None,
            'all_candidates': [],
        }

    # 2단계: 모호 매칭 (character level)
    # v12 (PR13): fetch 풀을 FETCH_TOP_N으로 늘려 정답 누락 방지.
    # _atc_tiebreak에서 fuzzy_top_n(=5)으로 다시 잘라 외부 시그니처는 그대로 유지.
    fuzzy_results = fuzzy_match(query, df, top_n=FETCH_TOP_N)
    # v9: jamo trigger는 원본 fuzzy 점수 기준 유지 (tiebreak는 순서만 변경)
    _orig_top_fuzzy_score = fuzzy_results[0]['score'] if fuzzy_results else 0

    # 2.2단계 (v9 PR12): ATC 기반 tiebreak (fuzzy)
    fuzzy_results = _atc_tiebreak(fuzzy_results, ATC_MAP, normalized_query, fuzzy_top_n)
    top_fuzzy_score = fuzzy_results[0]['score'] if fuzzy_results else 0

    # 2.5단계 (v4): fuzzy 원본 점수가 낮으면 자모 분해 fuzzy 추가 시도
    if _orig_top_fuzzy_score < JAMO_TRIGGER_THRESHOLD:
        # v12 (PR13): jamo도 동일하게 fetch 풀 확장
        jamo_results = fuzzy_match_jamo(query, df, top_n=FETCH_TOP_N)
        # 2.7단계 (v9 PR12): jamo 결과에도 ATC tiebreak
        jamo_results = _atc_tiebreak(jamo_results, ATC_MAP, normalized_query, fuzzy_top_n)
        if jamo_results:
            top_jamo_score = jamo_results[0]['score']
            if top_jamo_score > top_fuzzy_score and top_jamo_score >= JAMO_MIN_ACCEPT_SCORE:
                return {
                    'query': query,
                    'match_type': 'fuzzy_jamo',
                    'confidence': top_jamo_score,
                    'best_match': jamo_results[0]['drug_info'],
                    'all_candidates': [c['drug_info'] for c in jamo_results],
                }

    # 2단계 결과 사용
    if not fuzzy_results:
        return {
            'query': query,
            'match_type': 'fuzzy',
            'confidence': 0,
            'best_match': None,
            'all_candidates': [],
        }

    top_score = fuzzy_results[0]['score']
    return {
        'query': query,
        'match_type': 'fuzzy',
        'confidence': top_score,
        'best_match': fuzzy_results[0]['drug_info'] if top_score >= 70 else None,
        'all_candidates': [c['drug_info'] for c in fuzzy_results],
    }


print('match_drug 통합 함수 정의 완료')


## 매칭 결과 enrichment 함수

`match_drug` 반환값의 `best_match`에 `drug_info_detail`(38열, 성분·효능·용법 등)과
`e약은요정보`(15열, 일반인 친화 설명)를 `품목일련번호` 기준으로 머지한다.

후속 노트북(03_dur, 04_medication_guide)에서 사용자에게 노출할 정보를 한 번에 모으는 단계.

**반환 형식**: `match_drug` 결과 dict + 키 2개 추가
- `detail`: dict | None — 품목 상세 정보 (성분·효능 등)
- `easy`: dict | None — e약은요정보 (대중 친화 설명)
- 매칭 실패(`best_match=None`) 또는 join 키 없으면 두 값 모두 None

In [ ]:
def enrich_match(match_result: dict, df_detail: pd.DataFrame, df_easy: pd.DataFrame) -> dict:
    """match_drug 결과의 best_match에 detail/easy 정보를 머지한다.

    품목일련번호를 키로 df_detail/df_easy에서 추가 정보를 조회.
    매칭 실패(best_match=None)이면 detail/easy도 None.

    Args:
        match_result: match_drug() 반환값
        df_detail: 2_item_permit_detail 데이터프레임 (성분·효능·용법 등)
        df_easy: 5_easy_excel 데이터프레임 (e약은요정보)

    Returns:
        match_result + 'detail', 'easy' 키 추가된 dict
    """
    enriched = {**match_result, 'detail': None, 'easy': None}

    best = match_result.get('best_match')
    if best is None:
        return enriched

    item_id = best.get('품목일련번호')
    if item_id is None:
        return enriched

    detail_rows = df_detail[df_detail['품목일련번호'] == item_id]
    if len(detail_rows) > 0:
        enriched['detail'] = detail_rows.iloc[0].to_dict()

    easy_rows = df_easy[df_easy['품목일련번호'] == item_id]
    if len(easy_rows) > 0:
        enriched['easy'] = easy_rows.iloc[0].to_dict()

    return enriched


# enrichment 인라인 데모
print('enrichment 데모 (대표 2건):')
for q in ['게보린정', '닥터베아제정']:
    result = match_drug(q, df_drug)
    enriched = enrich_match(result, df_detail, df_easy)
    print(f"\n  '{q}'")
    if enriched['best_match']:
        print(f"    품목명: {enriched['best_match']['품목명']}")
    print(f"    detail 매칭됨: {enriched['detail'] is not None}")
    print(f"    easy 매칭됨: {enriched['easy'] is not None}")
    # easy에 보통 '이 약은 어떻게 사용합니까?' 컬럼이 있어 일부 미리보기
    if enriched['easy']:
        for col in ['이 약의 효능은 무엇입니까?', '이 약은 어떻게 사용합니까?']:
            val = enriched['easy'].get(col)
            if val and not (isinstance(val, float) and pd.isna(val)):
                preview = str(val)[:100].replace('|', ' | ')
                print(f"    easy.{col}: {preview}...")

## 실제 약품명 매칭 테스트

8개 시나리오로 검증:

1. 일반 약 정확 매칭 (`타이레놀정500mg`)
2. 공백 포함 (`타이레놀정 500mg`)
3. 유명 약 정확 매칭 (`게보린정`)
4. 만성질환 약 (`아모잘탄정 5/50mg`)
5. 부분 입력 - 모호 매칭 (`타이레놀`)
6. 부분 입력 - 모호 매칭 (`게보린`)
7. 존재하지 않는 약 (`존재하지않는약XYZ`)
8. 일반의약품 (`닥터베아제정`)

In [ ]:
TEST_QUERIES = [
    # 정확 매칭 의도
    '타이레놀정500mg',
    '타이레놀정 500mg',
    '게보린정',
    '아모잘탄정 5/50mg',
    '닥터베아제정',
    # 모호 매칭 의도 (부분 입력)
    '타이레놀',
    '게보린',
    # 추가 케이스 (개선 검증)
    'Tylenol',                       # 영문 입력
    '게부린정',                       # 오타 (보 -> 부)
    '아모잘탄플러스정 5/100/12.5mg',  # 복합제
    '',                              # 빈 문자열 edge case
    # 매칭 실패 의도
    '존재하지않는약XYZ',
]

for query in TEST_QUERIES:
    result = match_drug(query, df_drug)
    print('=' * 80)
    print(f"입력: '{query}'")
    print(f"매칭 유형: {result['match_type']} | 신뢰도: {result['confidence']}")

    if result['best_match'] is None:
        print('  [FAIL] 매칭 실패')
        continue

    best = result['best_match']
    print(f"  1순위: {best['품목명']}")
    print(f"  업체: {best['업체명']}")
    print(f"  품목일련번호: {best['품목일련번호']}")

    if result['match_type'] == 'exact_multiple':
        print(f"  [WARN] 동명이품 {len(result['all_candidates'])}개")
    elif result['match_type'] == 'fuzzy' and len(result['all_candidates']) > 1:
        print(f"  Top 3 후보:")
        for i, c in enumerate(result['all_candidates'][:3], 1):
            print(f"    {i}. {c['품목명']} ({c['업체명']})")

    # enrichment 가용 여부 (간단히)
    enriched = enrich_match(result, df_detail, df_easy)
    print(f"  enrichment: detail={enriched['detail'] is not None}, easy={enriched['easy'] is not None}")


# ====================================================================
# 평가 메트릭 (PR5 v5/PR7 추가)
#   - 골든셋(EVAL_SET)에 정답 품목일련번호를 라벨링하고
#     MRR / P@1 / P@5 + case_type별 세분 지표로 매칭 성능을 측정한다.
#   - 이후 PR에서 이 셀을 다시 실행하면 회귀 여부가 숫자로 드러난다.
# ====================================================================

from collections import defaultdict


# 골든셋
#   - expected_item_no는 v1~v4 PyCharm 실행으로 검증된 정답.
#   - expected_item_no=None이면 의도적 실패 케이스 (best_match=None이 정답).
#   - 사용자가 실제 데이터로 정답 확인 후 자유롭게 확장 가능.
EVAL_SET = [
    # ============================================================
    # v5 (기존 12건) - baseline 100% 통과 케이스
    # ============================================================

    # exact: 정규화 후 DB와 정확 일치
    {'query': '타이레놀정500mg',                  'expected_item_no': '202106092', 'case_type': 'exact',         'note': '정규화 후 정확'},
    {'query': '타이레놀정 500mg',                 'expected_item_no': '202106092', 'case_type': 'exact',         'note': '공백 포함'},
    {'query': '게보린정',                         'expected_item_no': '197900277', 'case_type': 'exact',         'note': '괄호 안 정보 제거'},
    {'query': '아모잘탄정 5/50mg',                'expected_item_no': '200902350', 'case_type': 'exact',         'note': '슬래시 함량'},
    {'query': '닥터베아제정',                     'expected_item_no': '200300406', 'case_type': 'exact',         'note': '단순'},
    {'query': '아모잘탄플러스정 5/100/12.5mg',    'expected_item_no': '201705043', 'case_type': 'exact',         'note': '복합 함량'},

    # prefix: 짧은 query -> 본형 매칭
    {'query': '타이레놀',                         'expected_item_no': '202106092', 'case_type': 'prefix',        'note': '브랜드만 -> 본형'},
    {'query': '게보린',                           'expected_item_no': '197900277', 'case_type': 'prefix',        'note': '브랜드만 -> 본형 (수출명 무시)'},

    # fuzzy_jamo: 한국어 1자모 오타
    {'query': '게부린정',                         'expected_item_no': '197900277', 'case_type': 'fuzzy_jamo',    'note': '부->보 (모음 1자모 오타)'},

    # fail_expected: 의도적 실패 (best_match=None이 정답)
    {'query': 'Tylenol',                          'expected_item_no': None,        'case_type': 'fail_expected', 'note': '영문 입력 (별도 PR 예정)'},
    {'query': '존재하지않는약XYZ',                'expected_item_no': None,        'case_type': 'fail_expected', 'note': 'DB에 없는 약'},
    {'query': '',                                 'expected_item_no': None,        'case_type': 'fail_expected', 'note': '빈 입력'},

    # ============================================================
    # v6 추가: probe2 발굴 일반 케이스 12건
    # ============================================================

    # fuzzy_jamo 보강 (자모 오타 다양한 패턴)
    {'query': '타이래놀',                         'expected_item_no': '202106092', 'case_type': 'fuzzy_jamo',    'note': '레->래 모음 오타 (현재 우먼스타이레놀 1순위 오답)'},
    {'query': '닥타베아제정',                     'expected_item_no': '200300406', 'case_type': 'fuzzy_jamo',    'note': '터->타 모음 오타 (현재 베아제정 1순위 오답)'},
    {'query': '겨보린정',                         'expected_item_no': '197900277', 'case_type': 'fuzzy_jamo',    'note': '게->겨 큰 모음 변화'},

    # fuzzy 신규 (format edge / 변종 의도)
    {'query': '타이레놀500밀리그램',              'expected_item_no': '202106092', 'case_type': 'fuzzy',         'note': '한글 단위 표기 (정 누락)'},
    {'query': '타이레놀500',                      'expected_item_no': '202106092', 'case_type': 'fuzzy',         'note': '정/mg 누락'},
    {'query': '타이레놀콜드',                     'expected_item_no': '202106954', 'case_type': 'fuzzy',         'note': '변종 명시 -> 변종이 1순위 (정상 동작 검증)'},
    {'query': '타이레놀이알',                     'expected_item_no': '202200407', 'case_type': 'fuzzy',         'note': '서방형 ER 의도 (현재 본형 1순위 오답, fetch_n=15 한계)'},

    # prefix 신규 (brand-only OTC)
    {'query': '베아제',                           'expected_item_no': '198700405', 'case_type': 'prefix',        'note': '대웅 소화제 브랜드명만'},
    {'query': '우루사',                           'expected_item_no': '198100119', 'case_type': 'prefix',        'note': '대웅 간장약, 100mg 본형'},
    {'query': '판피린',                           'expected_item_no': '199400202', 'case_type': 'prefix',        'note': '동아 코감기약, T정 본형'},

    # fail_expected 신규 (성분명 입력, 단일 정답 정의 불가)
    {'query': '아세트아미노펜',                   'expected_item_no': None,        'case_type': 'fail_expected', 'note': '성분명 (DB 198건). 별도 성분명 검색 기능 필요'},
    {'query': '암로디핀',                         'expected_item_no': None,        'case_type': 'fail_expected', 'note': '성분명 (DB 562건). 위와 동일'},

    # ============================================================
    # v6 추가: probe3 발굴 만성질환 6건 (당뇨 3 + 고혈압 3, 프로젝트 타깃)
    # ============================================================

    # [당뇨]
    {'query': '자누비아정 100mg',                 'expected_item_no': '200710759', 'case_type': 'exact',         'note': '당뇨/시타글립틴 (MSD/종근당), normalize 후 정확'},
    {'query': '메트포르민정 500mg',               'expected_item_no': None,        'case_type': 'fail_expected', 'note': '당뇨 성분명+용량 (DB 628건). 회사 모호'},
    {'query': '다이아벡스정',                     'expected_item_no': '198500321', 'case_type': 'fuzzy',         'note': '당뇨/메트포르민 (대웅), mg 누락 -> 임상 표준 500mg 본형 채택 (250/1000mg 변종 후순위 기대)'},

    # [고혈압]
    {'query': '노바스크정 5mg',                   'expected_item_no': '200610660', 'case_type': 'exact',         'note': '고혈압/암로디핀 (Pfizer), normalize 후 정확'},
    {'query': '디오반정 80mg',                    'expected_item_no': '200108763', 'case_type': 'fuzzy',         'note': '고혈압/발사르탄 (Novartis), DB명은 디오반필름코팅정80밀리그램 - 필름코팅 누락'},
    {'query': '코자정',                           'expected_item_no': '200811814', 'case_type': 'exact',         'note': '고혈압/로사르탄 (MSD), 코자정(로사르탄칼륨) normalize 후 정확'},
]


def _find_rank(candidates: list, expected_item_no) -> int | None:
    """all_candidates에서 expected_item_no의 1-based rank를 찾는다.

    품목일련번호 컬럼 타입이 일관되지 않을 수 있어 문자열 비교.

    Args:
        candidates: match_drug 결과의 'all_candidates' (list of dict).
        expected_item_no: 정답 품목일련번호.

    Returns:
        rank (1-based, 1=top) | None (정답 미포함 또는 expected가 None).
    """
    if not candidates or expected_item_no is None:
        return None
    target = str(expected_item_no)
    for i, c in enumerate(candidates, start=1):
        if str(c.get('품목일련번호', '')) == target:
            return i
    return None


def evaluate(eval_set: list, match_drug_fn, df, verbose: bool = True) -> dict:
    """골든셋으로 매칭 성능을 평가한다.

    메트릭:
        - MRR (Mean Reciprocal Rank): 정답 rank의 역수 평균.
          fail_expected에서 best_match=None이면 1.0으로 카운트 (의도대로 실패).
        - P@1: 1순위에 정답이 있는 비율.
        - P@5: top 5 안에 정답이 있는 비율.
        - by_case_type: case_type별 세분.

    Args:
        eval_set: 골든셋 리스트 (each: dict with query/expected_item_no/case_type/note).
        match_drug_fn: 평가할 매칭 함수 (match_drug 시그너처와 동일).
        df: drug_info 데이터프레임.
        verbose: True면 콘솔 표 출력.

    Returns:
        {'total', 'MRR', 'P@1', 'P@5', 'by_case_type', 'failures'} dict.
    """
    results = []
    failures = []
    by_type = defaultdict(list)

    for entry in eval_set:
        q = entry['query']
        expected = entry['expected_item_no']
        case_type = entry['case_type']

        match_result = match_drug_fn(q, df)
        best_match = match_result.get('best_match')

        if case_type == 'fail_expected':
            # best_match=None이 정답
            rank = 1 if best_match is None else None
        else:
            rank = _find_rank(match_result.get('all_candidates', []), expected)

        reciprocal = (1.0 / rank) if rank else 0.0

        record = {
            'query': q,
            'case_type': case_type,
            'expected_item_no': expected,
            'actual_item_no': str(best_match.get('품목일련번호', '')) if best_match else None,
            'actual_name': best_match.get('품목명') if best_match else None,
            'match_type': match_result.get('match_type'),
            'confidence': match_result.get('confidence'),
            'rank': rank,
            'reciprocal_rank': reciprocal,
            'note': entry.get('note', ''),
        }
        results.append(record)
        by_type[case_type].append(record)

        if rank != 1:
            failures.append(record)

    def _agg(records: list) -> dict:
        n = len(records)
        if n == 0:
            return {'count': 0, 'MRR': 0.0, 'P@1': 0.0, 'P@5': 0.0}
        return {
            'count': n,
            'MRR': sum(r['reciprocal_rank'] for r in records) / n,
            'P@1': sum(1 for r in records if r['rank'] == 1) / n,
            'P@5': sum(1 for r in records if r['rank'] and r['rank'] <= 5) / n,
        }

    summary = {
        'total': len(results),
        **_agg(results),
        'by_case_type': {k: _agg(v) for k, v in by_type.items()},
        'failures': failures,
    }

    if verbose:
        print('\n' + '=' * 80)
        print('평가 결과 (EVAL_SET)')
        print('=' * 80)
        print(f"총 {summary['total']}건  "
              f"MRR={summary['MRR']:.3f}  "
              f"P@1={summary['P@1']:.1%}  "
              f"P@5={summary['P@5']:.1%}")
        print()
        print(f"{'case_type':<16} {'n':>3}  {'MRR':>6}  {'P@1':>6}  {'P@5':>6}")
        print('-' * 50)
        for ct, m in summary['by_case_type'].items():
            print(f"{ct:<16} {m['count']:>3}  {m['MRR']:>6.3f}  {m['P@1']:>6.1%}  {m['P@5']:>6.1%}")
        print()
        if failures:
            print(f"실패 케이스 ({len(failures)}건):")
            for f in failures:
                expected_str = f['expected_item_no'] or '(매칭 실패가 정답)'
                actual_str = f['actual_item_no'] or '(매칭 실패)'
                rank_str = f['rank'] if f['rank'] is not None else 'X'
                print(f"  [{f['case_type']:<14}] '{f['query']}'")
                print(f"      기대: {expected_str}  실제: {actual_str} (rank={rank_str})")
                print(f"      매칭명: {f['actual_name']}  match_type: {f['match_type']}  conf: {f['confidence']}")
        else:
            print('실패 케이스 없음. 전 케이스 통과.')

    return summary


# 평가 실행
eval_summary = evaluate(EVAL_SET, match_drug, df_drug, verbose=True)


# ============================================================
# v7 추가, v9 개선: Baseline JSON dump (회귀 비교 인프라)
# ============================================================
# VERSION_LABEL은 매 PR마다 한 줄만 갱신 (v9 → v10 → ...)
import json as _json_v7
from pathlib import Path as _Path_v7

VERSION_LABEL = 'v12'  # ← 매 PR마다 갱신
_OUT_DIR = _Path_v7('eval_results')
_OUT_DIR.mkdir(exist_ok=True)
_OUT_PATH = _OUT_DIR / f'{VERSION_LABEL}_baseline.json'

with open(_OUT_PATH, 'w', encoding='utf-8') as _f:
    _json_v7.dump(eval_summary, _f, ensure_ascii=False, indent=2, default=str)

print(f'\n[OK] 평가 결과 저장: {_OUT_PATH}')
print(f'  - VERSION_LABEL = {VERSION_LABEL!r} (매 PR마다 갱신)')
print('  - 이후 PR 결과를 vN_baseline.json으로 dump하면 diff로 회귀 자동 비교 가능')

## 작업 마무리 및 다음 단계

### 본 노트북에서 구현·검증한 사항

- 정규화 함수 (`normalize_drug_name`) — 괄호 제거, 단위 통일(밀리그램→mg 등), 공백·특수문자 제거
- 자모 분해 (`jamo_decompose`) — 한글 호환 자모 분해로 1~2글자 오타 보정
- 정확 매칭 (`exact_match`) — `_normalized` 컬럼 1:1 lookup
- 모호 매칭 (`fuzzy_match`) — rapidfuzz WRatio, FETCH_TOP_N=15 후보 풀
- 성분명 차단 정책 (`is_likely_ingredient_query`) — 단독 성분명 입력 시 임의 제품 매칭 차단
- ATC tiebreak (`_atc_tiebreak`) — 동일 ATC 후보 중 chunk + prefix×20 bonus 재정렬
- 통합 매칭 (`match_drug`) — exact → prefix → ingredient_blocked → fuzzy → jamo fuzzy 폴백
- enrichment (`enrich_match`) — 품목일련번호 기반 detail·easy 머지

### 평가 결과 (30건 골든셋, v12)

| 지표 | 값 |
|---|---:|
| P@1 (1순위 정답률) | 96.7% |
| P@5 (Top 5 정답률) | 100.0% |
| MRR (Mean Reciprocal Rank) | 0.983 |

case_type 별 분포:

| case_type | n | MRR | P@1 | P@5 |
|---|---:|---:|---:|---:|
| exact | 9 | 1.000 | 100% | 100% |
| prefix | 5 | 1.000 | 100% | 100% |
| fuzzy | 6 | 1.000 | 100% | 100% |
| fuzzy_jamo | 4 | 0.875 | 75% | 100% |
| fail_expected | 6 | 1.000 | 100% | 100% |

### 버전별 변화 (v6 → v12)

| 버전 | P@1 | 핵심 변경 |
|---|---:|---|
| v6 | 76.7% | 30건 골든셋 baseline 구성 |
| v8 | 86.7% | 성분명 단독 입력 차단 |
| v10 | 90.0% | ATC tiebreak 도입 |
| v11 | 93.3% | prefix×20 bonus |
| v12 | 96.7% | fetch_n 15 확장 |

### 참고 — 선행 연구 (Peters et al. 2011, NLM RxNorm)

NLM의 약품명 매칭 알고리즘 연구로, 정확 매칭 실패 시 다단계 보강 절차를 적용하는 접근법을 제시하였다. 본 노트북은 이러한 다단계 폴백 설계 철학을 참조하였으며, 각 단계의 알고리즘은 한국어 약품명 특성(자모 분해, 본형/변종 구분 등)에 맞게 재구성된 형태로 적용하였다.

Peters 원문이 보고한 평가 수치는 다음과 같다(참고용).

| 데이터셋 | 규모 | Top-1 | Top-3 |
|---|---:|---:|---:|
| Peters 2011 Surescripts | 2,679 | 67.9% | 90.0% |
| Peters 2011 Development | 5,566 | 76.8% | 92.9% |
| Peters 2011 MEDID | 10,266 | 84.8% | 96.2% |

 Peters는 Top-3까지, 본 노트북은 Top-5까지 보고하므로 상위 K 범위가 달라 직접 비교에 한계가 있다. 데이터셋 규모(수천 건 vs 30건)와 도메인(영문 약품명 vs 한국 약품명)에도 차이가 있다. 또한 본 노트북의 30건은 알고리즘 동작 검증용 골든셋이라는 점도 함께 고려할 필요가 있다.

### enrichment 커버리지

| 데이터 | 행수 | 커버리지 |
|---|---:|---|
| detail (#2) | 43,276 | 거의 전 약품 |
| easy (#5) | 4,762 | 일반의약품 위주 (약 11%) |

처방약(전문의약품)은 detail만 매칭됨. easy=False가 정상.

### 본 PR 범위 외 — 향후 개선

- 타이래놀(단일/복합제 ATC 갈라짐) 해결: ATC family prefix 매칭 또는 단일성분 우선 룰
- 영문→한글 brand 사전: 'Tylenol' → 한글 본형 cross-script 매핑
- 평가 셋 확장: HIRA 다빈도 약물 시드 + 가상 처방전 데이터 활용 검토 (규모는 팀 협의 후 결정). 실제 OCR 처방전 데이터를 사용할 경우 보건의료데이터 활용 가이드라인 및 IRB·개인정보보호 검토가 선행될 필요가 있어, 본 PR 범위에서는 다루지 않는다.
- 매칭 결과 캐싱: `functools.lru_cache` 등

### 다음 노트북 (03_dur_safety_check)

- 매칭된 `품목일련번호`를 키로 DUR 데이터 조회
- 4겹 안전 검증:
  - 동일성분 중복 (DUR 효능군 유형)
  - 효능군 중복 (DUR 효능군 유형)
  - 노인주의 (DUR 주의 유형, 정보 제공 형태)
  - 회수약 검출 (DUR 외 별도 안전 체크)
- HIRA(2019) 의원급 외래 처방 분포(표 4-31) 기준 상위 3개 DUR 유형(동일성분 중복 42.2% + 효능군 중복 35.4% + 노인주의 14.7%)을 우선 구현
- `enrich_match` 결과 그대로 활용 가능